### GRPO Trining 

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install flash-attn --no-build-isolation

In [ ]:
import json
import torch
import re
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

# 1. Load the SFT-tuned model
model_path = "sre_model_sft_final" # The folder created by the SFT notebook
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 2. Add LoRA (Specializing for policy exploration)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_v_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    use_gradient_checkpointing = "unsloth",
)

# 3. Environment-Based Reward Logic
from agents.split_brain.environment import SplitBrainEnv

def reward_fn_env(prompts, completions, task_id, **kwargs):
    """Executes the model's JSON action in the live environment."""
    rewards = []
    env = SplitBrainEnv()
    
    for completion, task in zip(completions, task_id):
        # Extract JSON from the completion
        match = re.search(r'\{.*\}', completion, re.DOTALL)
        if not match:
            rewards.append(-2.0) # Penalty for not producing JSON
            continue
            
        try:
            action_dict = json.loads(match.group(0))
            env.reset(task=task)
            # Execute the action in the env
            res = env.step(action_dict)
            
            # Reward is based on Global Health restoration
            # health 1.0 = +2.0, health 0.0 = -1.0
            score = (res.observation.global_health * 3.0) - 1.0
            rewards.append(score)
        except:
            rewards.append(-1.5) # Penalty for invalid JSON parameters
            
    return rewards

def reward_fn_format(completions, **kwargs):
    """Rewards models for using <think> tags and JSON blocks."""
    rewards = []
    for content in completions:
        score = 0.0
        if "<think>" in content and "</think>" in content: score += 0.5
        if "{" in content and "}" in content: score += 0.5
        rewards.append(score)
    return rewards

# 4. Prepare RL Dataset
with open("rl_prompts.json", "r") as f:
    raw_prompts = json.load(f)

# GRPO expects columns: 'prompt' and any metadata (task_id)
dataset = Dataset.from_list(raw_prompts)

# 5. Training Config (Optimized for 48GB VRAM)
training_args = GRPOConfig(
    use_vllm = False, # Set to True if you have vLLM installed for faster rollouts
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    bf16 = True,
    per_device_train_batch_size = 1, # GRPO expands this by num_generations
    gradient_accumulation_steps = 4,
    num_generations = 8, # Pushed to 8 generations per prompt for better variance
    max_prompt_length = 1024,
    max_completion_length = 512,
    num_train_epochs = 1,
    save_steps = 50,
    max_grad_norm = 0.1,
    output_dir = "outputs_grpo",
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [reward_fn_env, reward_fn_format],
    args = training_args,
    train_dataset = dataset,
)

# 6. Run Training
trainer.train()

# 7. Final Model Save
model.save_pretrained_merged("sre_model_final_v1", tokenizer, save_method = "merged_16bit")
print("GRPO Training Complete. Final '70B-Level' model saved.")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

os.makedirs("plots", exist_ok=True)

# Extract history
history = trainer.state.log_history
df = pd.DataFrame(history)

# GRPO logs can be messy; we filter for steps with reward data
df_rl = df[df['reward'].notna()].copy()

# Apply smoothing (rolling mean) to visualize trends clearly
window = 3
df_rl['smoothed_reward'] = df_rl['reward'].rolling(window=window).mean()

plt.figure(figsize=(15, 10))

# Plot 1: Total Mean Reward (The "Hockey Stick")
plt.subplot(2, 2, 1)
plt.plot(df_rl['step'], df_rl['reward'], color='#94a3b8', alpha=0.3, label="Raw")
plt.plot(df_rl['step'], df_rl['smoothed_reward'], color='#6366f1', linewidth=2.5, label="Smoothed")
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.title("GRPO Total Mean Reward")
plt.xlabel("Step")
plt.ylabel("Reward Score")
plt.legend()
plt.grid(True, alpha=0.2)

# Plot 2: Breakdown of Reward Functions
plt.subplot(2, 2, 2)
if 'rewards/reward_fn_env/mean' in df_rl.columns:
    plt.plot(df_rl['step'], df_rl['rewards/reward_fn_env/mean'], color='#10b981', label="Env (Health)")
if 'rewards/reward_fn_format/mean' in df_rl.columns:
    plt.plot(df_rl['step'], df_rl['rewards/reward_fn_format/mean'], color='#f59e0b', label="Format (<think>)")
plt.title("Reward Component Progress")
plt.xlabel("Step")
plt.ylabel("Component Score")
plt.legend()
plt.grid(True, alpha=0.2)

# Plot 3: KL Divergence (Stability Check)
# Target should be between 0.01 and 0.1 for healthy learning
plt.subplot(2, 2, 3)
plt.plot(df_rl['step'], df_rl['kl'], color='#ef4444', linewidth=2)
plt.title("KL Divergence\n(Policy Deviation Stability)")
plt.xlabel("Step")
plt.ylabel("KL Value")
plt.grid(True, alpha=0.2)

# Plot 4: Completion Length (Conciseness)
plt.subplot(2, 2, 4)
plt.plot(df_rl['step'], df_rl['completion_length'], color='#8b5cf6', linewidth=2)
plt.title("Average Completion Length\n(Reducing Yapping)")
plt.xlabel("Step")
plt.ylabel("Tokens")
plt.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("plots/grpo_training_inferences.png", dpi=300)
plt.show()

print("GRPO plots saved as plots/grpo_training_inferences.png")